# G01 — MIL-006 Qwen3-1.7B Generator QLoRA

Local-only RAG-SFT over checksum-validated official train answers and frozen evidence. OQ-003 remains unresolved, so this notebook must never transfer real data to Modal.

In [ ]:
# G02 — safe central configuration
RUN_MODE = "dry_run"  # dry_run | smoke | full
ALLOW_NETWORK = False
ALLOW_GPU = False
ALLOW_MODAL_REAL_DATA = False
ALLOW_FINETUNE = False
CONFIRM_REMOTE_EXECUTION = False
EXECUTION_BACKEND = "local"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"

In [ ]:
# G03 — authorization preflight
import json
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent.parent.resolve()
authorization = json.loads(
    (ROOT / "artifacts/reports/final/training-authorization.v1.json").read_bytes()
)
decision = authorization["decisions"]["FT-GENERATOR"]
assert decision["can_proceed"] is True and decision["backend"] == "local"
assert ALLOW_NETWORK is False and ALLOW_MODAL_REAL_DATA is False and EXECUTION_BACKEND == "local"

In [ ]:
# G04/G05 — immutable inputs and RAG-SFT provenance audit
from legal_rag.training.generator_qlora import load_generator_rows

DATASET = ROOT / "artifacts/training/rag-sft/v1"
PROVENANCE = DATASET / "training-examples.v1.jsonl"
MATERIAL = DATASET / "training-material.v1.jsonl"
MANIFEST = DATASET / "manifest.v1.json"
rows = load_generator_rows(PROVENANCE.read_bytes(), MATERIAL.read_bytes(), MANIFEST.read_bytes())
assert rows and all(row.evidence_ids for row in rows)
{"validated_rows": len(rows), "raw_text_displayed": False}

In [ ]:
# G06/G07 — aggregate-only stats and deterministic answer-only template
from legal_rag.training.recipes import GENERATOR_QLORA_CENTRAL, recipe_checksum

{
    "rows": len(rows),
    "evidence_ids": sum(len(row.evidence_ids) for row in rows),
    "recipe_checksum": recipe_checksum(GENERATOR_QLORA_CENTRAL),
    "loss_scope": GENERATOR_QLORA_CENTRAL.loss_scope,
}

## G08 / G09 / G10 — guarded model load, NF4/QLoRA, deterministic collator

Model loading is inside the guarded runner. Quantization does not reduce BTC parameter accounting; the runner checks base + adapter against the whole-system <4B gate.

In [ ]:
# G11/G12 — bounded smoke or one central full run
from legal_rag.training.generator_qlora import GeneratorTrainingConfig, run_generator_qlora

if RUN_MODE in {"smoke", "full"}:
    assert ALLOW_GPU is True and ALLOW_FINETUNE is True
    checkpoint = ROOT / ".local/models/qwen3-1.7b" / MODEL_REVISION
    output = ROOT / ".local/runs" / ("G3-notebook-" + RUN_MODE)
    run_report = run_generator_qlora(rows, GeneratorTrainingConfig(RUN_MODE, checkpoint, output))
else:
    run_report = {"state": "dry_run_complete", "validated_rows": len(rows)}
run_report

## G13 / G14 / G15 / G16 — evaluation, adapter, report, promotion

Run deterministic generation evaluation with fixed retrieval before promotion. The ignored adapter and run report are saved under `.local/runs`. Promotion remains blocked until METEOR/ROUGE-L, grounding, determinism, latency and EVAL-005 guards are compared against G1/G2. No DPO/RL/GRPO is allowed.